In [ ]:
from google.cloud import bigquery
import warnings
import pandas as pd
from datasets import Dataset
from utils.hub_card import push_dataset_card
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

client = bigquery.Client(project="ads-599-final")

In [ ]:
# Medication records for admitted patients covering two phases:
#   1. ED boarding phase: Pyxis dispenses during the ED stay (in_er = True)
#   2. Hospital ward phase: eMAR administrations from ED departure up to ICU transfer (in_er = False)
#
# ICU gate uses icu.icustays (not transfers) — only patients under actual ICU clinical management.
# eMAR event_txt values mapped to four discrete actions:
#   administer_medication, start_medication, stop_medication, rate_change.
# Flush, topical, and documentation events excluded.

query_admitted_meds = """
WITH icu_times AS (
  SELECT
    hadm_id,
    CAST(MIN(intime) AS TIMESTAMP) AS icu_intime
  FROM `physionet-data.mimiciv_3_1_icu.icustays`
  GROUP BY hadm_id
),

pyxis_part AS (
  SELECT
    e.subject_id,
    e.stay_id,
    e.hadm_id,
    CAST(p.charttime AS TIMESTAMP) AS pyxis_charttime,
    p.med_rn        AS pyxis_med_rn,
    p.name          AS pyxis_medication,
    CAST(NULL AS STRING)    AS emar_id,
    CAST(NULL AS TIMESTAMP) AS emar_charttime,
    CAST(NULL AS STRING)    AS emar_medication,
    CAST(NULL AS STRING)    AS event_txt,
    CAST(NULL AS TIMESTAMP) AS scheduletime,
    TRUE            AS in_er
  FROM `physionet-data.mimiciv_ed.edstays` e
  JOIN `physionet-data.mimiciv_ed.pyxis` p
    ON e.stay_id = p.stay_id
  WHERE (e.disposition = 'ADMITTED' OR e.hadm_id IS NOT NULL)
    AND p.charttime BETWEEN e.intime AND e.outtime
),

emar_part AS (
  SELECT
    e.subject_id,
    e.stay_id,
    em.hadm_id,
    CAST(NULL AS TIMESTAMP) AS pyxis_charttime,
    CAST(NULL AS INT64)     AS pyxis_med_rn,
    CAST(NULL AS STRING)    AS pyxis_medication,
    em.emar_id,
    CAST(em.charttime AS TIMESTAMP) AS emar_charttime,
    em.medication           AS emar_medication,
    em.event_txt,
    CAST(em.scheduletime AS TIMESTAMP) AS scheduletime,
    FALSE           AS in_er
  FROM `physionet-data.mimiciv_3_1_hosp.emar` em
  JOIN `physionet-data.mimiciv_ed.edstays` e
    ON em.hadm_id = e.hadm_id
  LEFT JOIN icu_times icu
    ON em.hadm_id = icu.hadm_id
  WHERE (e.disposition = 'ADMITTED' OR e.hadm_id IS NOT NULL)
    AND em.event_txt IN (
      'Administered',
      'Partial Administered',
      'Delayed Administered',
      'Administered Bolus from IV Drip',
      'Administered in Other Location',
      'Started',
      'Restarted',
      'Stopped',
      'Stopped As Directed',
      'Stopped - Unscheduled',
      'Stopped in Other Location',
      'Rate Change'
    )
    AND CAST(em.charttime AS TIMESTAMP) < COALESCE(icu.icu_intime, TIMESTAMP('9999-01-01'))
)

SELECT * FROM pyxis_part
UNION ALL
SELECT * FROM emar_part
ORDER BY subject_id, stay_id, COALESCE(pyxis_charttime, emar_charttime)
"""

print("Running admitted meds query...")
df_admitted_meds = client.query(query_admitted_meds).to_dataframe()
print(f"Shape: {df_admitted_meds.shape}")
print(f"Unique patients: {df_admitted_meds['subject_id'].nunique():,}")
pyxis_rows = df_admitted_meds['in_er'].sum()
emar_rows = (~df_admitted_meds['in_er']).sum()
print(f"\nED (Pyxis) rows: {pyxis_rows:,}")
print(f"Ward (eMAR) rows: {emar_rows:,}")
df_admitted_meds.head()

In [ ]:
DESCRIPTION = (
    "Medication administration records for admitted patients, covering both the ED boarding phase "
    "and the subsequent hospital ward stay (pre-ICU). Derived from hosp.emar. "
    "event_txt values mapped to four discrete actions: administer_medication, start_medication, "
    "stop_medication, rate_change. Flush, topical, and documentation events excluded."
)

ds = Dataset.from_pandas(df_admitted_meds, split='meds_admit')
ds.push_to_hub("ADS599-Capstone/raw_data", config_name="meds_admitted", data_dir="stay_medications")
push_dataset_card("ADS599-Capstone/raw_data", config_name="meds_admitted", split="meds_admit", data_dir="stay_medications", description=DESCRIPTION)
print("Pushed meds_admitted to HuggingFace Hub.")